# Experimentos: ATP y fuerza externa

Estudia cómo la concentración de ATP y una carga opuesta afectan las tasas y los observables emergentes de muchas trayectorias simuladas.

> **Nota científica:** KineSIM es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

## Resultados de aprendizaje

Este laboratorio separa dos efectos: (1) la concentración de ATP modifica la tasa de asociación del nucleótido y (2) una carga opuesta modifica tasas seleccionadas mediante una ley tipo Bell. Aprenderás a distinguir una tasa elemental de un observable emergente, como la velocidad media de un conjunto de trayectorias.

**Alcance:** las constantes usadas son ilustrativas. Los resultados son predicciones del modelo reducido, no curvas fuerza–velocidad experimentales ni parámetros medidos de kinesina-1.

## Instalación desde PyPI

Este notebook utiliza la distribución publicada de **KineSIM**. No modifica `sys.path` ni importa código desde la carpeta `src/`.

- Nombre de distribución: `kinesim` (PyPI no distingue mayúsculas).
- Namespace de importación: `protein_motor_sim` (se conserva por compatibilidad).
- Versión fijada para reproducibilidad: `0.1.2`.

Ejecuta la siguiente celda con acceso a Internet. Si la versión todavía no aparece en PyPI, primero debe publicarse con Twine. Para probar una subida en TestPyPI, consulta las instrucciones de instalación de este README; no mezcles los índices de TestPyPI y PyPI en una misma resolución de dependencias.


In [ ]:
%pip install --quiet "kinesim[notebooks]==0.1.2"


## 1. Preguntas del experimento

1. ¿Cómo cambia la tasa de unión de ATP con la concentración?
2. ¿Cómo cambia el comportamiento simulado al aumentar una carga opuesta?

Separaremos la ley cinética elemental de los resultados emergentes del conjunto de simulaciones.

## Fundamento: asociación de ATP

Una asociación bimolecular se puede escribir como $K+ATP\rightarrow K\cdot ATP$. Si la concentración de ATP se mantiene constante, la cinética de la etapa de unión se expresa mediante una tasa pseudo-primer orden:

$$
k_{\mathrm{ATP}}=k_{\mathrm{on}}[ATP].
$$

En este notebook, $k_{\mathrm{on}}$ se expresa en M$^{-1}$ s$^{-1}$ y $[ATP]$ en mol/L; el resultado queda en s$^{-1}$. La conversión de mM a M debe hacerse antes de evaluar la ley: $1\,\mathrm{mM}=10^{-3}\,\mathrm{M}$.

La ecuación describe únicamente la unión de ATP. La velocidad del ciclo completo también depende de la hidrólisis, los pasos mecánicos y el desprendimiento.

In [ ]:
from importlib.metadata import version as distribution_version
print("KineSIM distribution:", distribution_version("kinesim"))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import replace

from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.kinetics.rate_laws import ATPBindingRate
from protein_motor_sim.kinetics.force_dependence import BellRate
from protein_motor_sim.kinetics.rate_laws import ConstantRate
from protein_motor_sim.simulation.ensemble import run_ensemble
from protein_motor_sim.analysis.statistics import summarize_trajectories
from protein_motor_sim.analysis.parameter_sweep import sweep_atp_concentration

## 2. Efecto directo de ATP sobre la tasa de asociación

Para una asociación bimolecular aproximada, la tasa pseudo-primer orden es `k_ATP = k_on [ATP]`. Esta relación describe solo la etapa de unión, no la velocidad global del motor.

In [ ]:
k_on = 1.0e6  # M^-1 s^-1, valor ilustrativo
rate_law = ATPBindingRate(association_rate_M_s=k_on)
atp_M = np.logspace(-6, -2, 100)
rates_s = np.array([rate_law(Environment(atp_concentration=c)) for c in atp_M])

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(atp_M * 1e3, rates_s)
ax.set_xlabel("[ATP] (mM)")
ax.set_ylabel("Tasa de asociación (s$^{-1}$)")
ax.set_title("Ley de asociación de ATP")
ax.grid(True, which="both", alpha=0.25)
plt.show()

### Interpretación de la primera gráfica

Se utiliza una escala logarítmica para representar varios órdenes de magnitud de concentración. Como $k_{\mathrm{ATP}}$ es proporcional a $[ATP]$, la relación ideal aparece como una recta de pendiente uno en una gráfica log–log.

El valor `k_on = 1e6 M^-1 s^-1` es ilustrativo. La recta comprueba la ley implementada; por sí sola no valida la cinética real de kinesina.

## 3. Barrido de concentración de ATP

Ejecutamos múltiples realizaciones para cada concentración. La media y desviación estándar describen la dispersión entre trayectorias, no la incertidumbre experimental.

### Diseño del barrido de concentración

El barrido ejecuta `n_runs=30` trayectorias por cada concentración. Para cada condición se calcula la velocidad media neta de cada trayectoria y después se obtiene la media y la desviación estándar entre realizaciones.

La desviación estándar describe la dispersión del conjunto simulado. No es el error estándar de la media ni un intervalo de confianza. `t_max` fija la ventana máxima de observación; una trayectoria puede terminar antes si alcanza el estado terminal `detached`.

In [ ]:
motor = Kinesin1()
simulator = GillespieSimulator()
base_environment = Environment(temperature_K=310.0, external_force_N=0.0)
concentrations_M = np.array([0.05, 0.1, 0.25, 0.5, 1.0, 2.0]) * 1e-3

atp_results = sweep_atp_concentration(
    simulator=simulator,
    motor=motor,
    environment=base_environment,
    concentrations_M=concentrations_M,
    t_max=0.7,
    n_runs=30,
    seed=120,
)
atp_df = pd.DataFrame(atp_results)
atp_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    atp_df["atp_concentration_M"] * 1e3,
    atp_df["mean_velocity_nm_s"],
    yerr=atp_df["std_velocity_nm_s"],
    marker="o", capsize=3,
)
ax.set_xlabel("[ATP] (mM)")
ax.set_ylabel("Velocidad media neta (nm/s)")
ax.set_title("Respuesta simulada frente a ATP")
ax.grid(alpha=0.25)
plt.show()

### Cómo interpretar la respuesta a ATP

Al aumentar ATP, aumenta la tasa de asociación en la primera etapa. Sin embargo, la velocidad global no tiene por qué aumentar linealmente en todo el rango: otras etapas del ciclo pueden convertirse en limitantes y el desprendimiento puede interrumpir trayectorias.

Lee la gráfica como una respuesta **del modelo y de sus parámetros actuales**. Para compararla con un experimento real habría que definir el mismo observable, usar constantes medidas y comprobar que la red represente adecuadamente el ciclo mecanoquímico.

## 4. Efecto de la fuerza opuesta

La clase `BellRate` modifica las tasas según la fuerza. En el modelo actual, la carga reduce la tasa de avance, aumenta la tasa de retroceso y puede aumentar el desprendimiento según la convención elegida en el modelo. Esto es una parametrización fenomenológica; no es todavía una curva fuerza–velocidad calibrada.

## Fundamento: dependencia de las tasas con la fuerza

La aproximación de Bell modifica una tasa de transición según la carga:

$$
k(F)=k_0\exp\left(-s\,\theta\,\frac{F\delta}{k_BT}\right).
$$

Aquí $k_0$ es la tasa sin carga, $F$ la fuerza externa, $\delta$ una distancia característica, $\theta$ una fracción efectiva de carga y $s$ el signo de sensibilidad configurado. En KineSIM, una fuerza positiva representa una carga opuesta al avance en $+x$.

En la red actual, `force_direction=+1` reduce la tasa de avance; `force_direction=-1` aumenta la tasa de retroceso y también la tasa de desprendimiento. Esta es una elección fenomenológica de parametrización. No es un ajuste termodinámico completo ni una ley universal para todas las transiciones de kinesina.

In [ ]:
forces_pN = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])
force_rows = []
for force_pN in forces_pN:
    condition = Environment(
        atp_concentration=1e-3,
        external_force_N=force_pN * 1e-12,
        temperature_K=310.0,
    )
    ensemble = run_ensemble(
        simulator, motor, condition, t_max=0.7, n_runs=30, seed=500 + int(force_pN * 10)
    )
    stats = summarize_trajectories(ensemble.trajectories)
    force_rows.append({
        "force_pN": force_pN,
        "mean_velocity_nm_s": stats.mean_velocity_nm_s,
        "std_velocity_nm_s": stats.std_velocity_nm_s,
        "mean_run_length_nm": stats.mean_run_length_nm,
    })
force_df = pd.DataFrame(force_rows)
force_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    force_df["force_pN"], force_df["mean_velocity_nm_s"],
    yerr=force_df["std_velocity_nm_s"], marker="o", capsize=3,
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Fuerza opuesta (pN)")
ax.set_ylabel("Velocidad media neta (nm/s)")
ax.set_title("Respuesta simulada frente a la carga")
ax.grid(alpha=0.25)
plt.show()

### Interpretación de la respuesta a la carga

La gráfica compara velocidades medias netas de conjuntos independientes bajo distintas fuerzas opuestas. Una disminución de la velocidad puede resultar de una menor tasa de avance, una mayor tasa de retroceso o cambios en la supervivencia de las trayectorias.

En este modelo, no se ha impuesto balance detallado local entre tasas directas e inversas, consumo de ATP y trabajo mecánico. Por tanto, no uses esta curva para extraer una fuerza de parada experimental ni una eficiencia energética real.

## 5. Interpretación y límites

- Aumentar ATP acelera la asociación de ATP, pero otras etapas pueden limitar el ciclo.
- La fuerza cambia tasas particulares según el modelo de Bell; no se ha impuesto una descripción termodinámica completa.
- La velocidad reportada es la velocidad neta de la trayectoria hasta el final de observación o desprendimiento.
- No interpretes estos resultados como parámetros experimentales ni como predicción cuantitativa de la kinesina real sin calibración.

## Qué se puede concluir y qué no

**Sí podemos concluir:** la ley de asociación implementada crece linealmente con ATP; la ley tipo Bell modifica exponencialmente las tasas seleccionadas; el conjunto de trayectorias permite observar la variabilidad inducida por el algoritmo estocástico.

**No podemos concluir todavía:** que la curva obtenida reproduzca datos experimentales de kinesina, que los valores de fuerza sean parámetros calibrados o que la dependencia de fuerza sea termodinámicamente completa.

## Referencias

- Block, S. M. (2007). *Kinesin motor mechanics: Binding, stepping, tracking, gating, and limping*. Biophysical Journal, 92(9), 2986–2995. [https://doi.org/10.1529/biophysj.106.100677](https://doi.org/10.1529/biophysj.106.100677)
- Carter, N. J., & Cross, R. A. (2005). *Mechanics of the kinesin step*. Nature, 435, 308–312. [https://doi.org/10.1038/nature03528](https://doi.org/10.1038/nature03528)